<a href="https://colab.research.google.com/github/pluplu68/agent-lab/blob/main/RAG/RAG_Retrieval_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Retrieval Lab — From Baselines to Evaluation

**GitHub-ready notebook · API-free core · CPU-friendly · English toy corpus**

**Goal:** Build and compare **BM25 → Dense Retrieval → Hybrid (RRF) → Cross-Encoder Reranking**, then evaluate retrieval and prepare a grounded-generation prompt.

**Run:** Execute cells top-to-bottom in Google Colab or Jupyter. **No paid API keys, accounts, or GPU required.** The first run needs internet to install packages and download two small pretrained models from Hugging Face. Subsequent runs use the model cache.

**What this is not:** A production search engine or a statistically meaningful benchmark. The tiny dataset and relevance labels are illustrative. The final local LLM generation step is optional.

| Stage | What it does | Implementation |
|---|---|---|
| Indexing | Clean, chunk, embed documents | Pre-chunked toy corpus + sentence-transformer |
| Retrieval | Find relevant documents | BM25, cosine similarity |
| Fusion | Combine ranked lists | Reciprocal Rank Fusion (RRF) |
| Reranking | Refine candidate ordering | CrossEncoder |
| Evaluation | Measure retrieval quality | Recall@K, MRR@K, nDCG@K |
| Generation | Answer with retrieved evidence | Grounded prompt; optional local FLAN-T5 |

## 0. Setup

Run once per new environment. This demo avoids LangChain version-specific wrappers, external vector databases, and paid LLM APIs. **BM25 is lexical search; dense retrieval is semantic search.** For a tiny corpus, exact vector dot products are sufficient (no FAISS server needed).

In [ ]:
%pip -q install "rank-bm25>=0.2.2" "sentence-transformers>=3.0" "pandas>=2.0" "numpy>=1.24" "sentencepiece>=0.2.0"

In [ ]:
import math
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 120)

## 1. Corpus and chunking

Each `doc_id` is a **retrieval unit (chunk)**. `parent_id` connects related chunks to a larger section. We deliberately keep IDs stable so evaluation labels remain valid. Replace `DOCS` with your own documents later.

In [ ]:
DOCS = [
    {"id": "d01", "parent_id": "basics", "text": "Retrieval-augmented generation (RAG) retrieves external evidence and gives it to a language model for grounded question answering."},
    {"id": "d02", "parent_id": "index", "text": "Chunking divides long documents into smaller coherent passages. Chunk size and overlap trade off retrieval specificity against missing context."},
    {"id": "d03", "parent_id": "retrieval", "text": "BM25 is a lexical sparse retriever that scores documents using exact query term matches, term frequency, and document length normalization."},
    {"id": "d04", "parent_id": "retrieval", "text": "Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables semantic matching beyond exact words."},
    {"id": "d05", "parent_id": "fusion", "text": "Reciprocal Rank Fusion (RRF) combines multiple ranked lists using the reciprocal of each document's rank. It does not need comparable raw scores."},
    {"id": "d06", "parent_id": "fusion", "text": "A cross-encoder reranker jointly reads a query and a candidate passage, predicts a relevance score, and reorders a small retrieved candidate set."},
    {"id": "d07", "parent_id": "evaluation", "text": "Recall at K measures how many relevant documents are retrieved; MRR measures the first relevant rank; nDCG at K rewards relevant documents near the top."},
    {"id": "d08", "parent_id": "index", "text": "Parent document retrieval searches short child chunks and returns the larger corresponding parent section, preserving surrounding context for generation."},
    {"id": "d09", "parent_id": "advanced", "text": "ColBERT stores contextualized token vectors. Its late-interaction MaxSim score takes each query token's highest similarity to a document token and sums the maxima."},
    {"id": "d10", "parent_id": "advanced", "text": "HyDE asks a language model to write a hypothetical answer passage and embeds that passage to retrieve real evidence without task-specific relevance labels."},
    {"id": "d11", "parent_id": "control", "text": "Corrective RAG (CRAG) evaluates retrieved evidence, keeps useful passages, and can supplement or replace weak evidence using an additional search source."},
    {"id": "d12", "parent_id": "control", "text": "Original Self-RAG trains a generator to produce retrieval and critique reflection tokens that guide when to retrieve and how to assess evidence and answers."},
    {"id": "d13", "parent_id": "query", "text": "Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve coverage when useful."},
    {"id": "d14", "parent_id": "query", "text": "Query decomposition splits a multi-part information need into a few independent subquestions; excessive decomposition can increase latency and redundancy."},
    {"id": "d15", "parent_id": "generation", "text": "Grounded generation instructs the LLM to answer only from retrieved sources, abstain when evidence is insufficient, and identify supporting document IDs."},
    {"id": "d16", "parent_id": "index", "text": "A vector index stores document embeddings to accelerate nearest-neighbor search. Exact numpy similarity is fine for small demos; ANN indexes scale better."},
    {"id": "d17", "parent_id": "advanced", "text": "RAPTOR clusters passages, summarizes groups recursively, and retrieves nodes at several abstraction levels for both local facts and broad questions."},
    {"id": "d18", "parent_id": "query", "text": "A query router selects a data source such as documents, SQL, or web search based on the question. A route must correspond to an implemented tool."},
]

DOC_BY_ID = {d["id"]: d for d in DOCS}
assert len(DOC_BY_ID) == len(DOCS)
pd.DataFrame(DOCS).head(5)

,id,parent_id,text
0,d01,basics,Retrieval-augmented generation (RAG) retrieves external evidence and gives it to a language model for grounded quest...
1,d02,index,Chunking divides long documents into smaller coherent passages. Chunk size and overlap trade off retrieval specifici...
2,d03,retrieval,"BM25 is a lexical sparse retriever that scores documents using exact query term matches, term frequency, and documen..."
3,d04,retrieval,Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables sema...
4,d05,fusion,Reciprocal Rank Fusion (RRF) combines multiple ranked lists using the reciprocal of each document's rank. It does no...


In [ ]:
def chunk_words(text: str, chunk_size: int = 30, overlap: int = 6) -> list[str]:
    """Simple word-based chunker for illustration; real projects may use sentence-aware splitting."""
    if not (0 <= overlap < chunk_size):
        raise ValueError("Require 0 <= overlap < chunk_size")
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), step)]

sample = DOCS[0]["text"] + " " + DOCS[1]["text"]
print("Example chunks:", chunk_words(sample, chunk_size=25, overlap=5))

Example chunks: ['Retrieval-augmented generation (RAG) retrieves external evidence and gives it to a language model for grounded question answering. Chunking divides long documents into smaller coherent passages.', 'documents into smaller coherent passages. Chunk size and overlap trade off retrieval specificity against missing context.']


## 2. BM25 and dense retrieval

**BM25:** exact token matches. **Dense:** normalized sentence embeddings + cosine similarity. `all-MiniLM-L6-v2` is an English, lightweight model; replace it with a multilingual encoder for Chinese documents.

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())  # suitable for English only

bm25 = BM25Okapi([tokenize(d["text"]) for d in DOCS])

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMBED_MODEL, device="cpu")

# Normalizing makes dot product equal cosine similarity.
doc_vectors = encoder.encode(
    [d["text"] for d in DOCS], normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
print(f"Indexed {len(DOCS)} chunks, embedding shape: {doc_vectors.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 18 chunks, embedding shape: (18, 384)


In [ ]:
def rank_scores(scores, top_k: int = 10, *, positive_only: bool = False) -> list[dict]:
    pairs = [(DOCS[i]["id"], float(s)) for i, s in enumerate(scores)]
    if positive_only:
        pairs = [(doc_id, s) for doc_id, s in pairs if s > 0]
    pairs.sort(key=lambda item: item[1], reverse=True)
    return [{"doc_id": doc_id, "score": score} for doc_id, score in pairs[:top_k]]


def bm25_search(query: str, top_k: int = 10) -> list[dict]:
    scores = bm25.get_scores(tokenize(query))
    return rank_scores(scores, top_k, positive_only=True)


def dense_search(query: str, top_k: int = 10) -> list[dict]:
    query_vector = encoder.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    return rank_scores(doc_vectors @ query_vector, top_k)

query = "How do I combine semantic and keyword search results?"
print("BM25 :", [r["doc_id"] for r in bm25_search(query, 3)])
print("Dense:", [r["doc_id"] for r in dense_search(query, 3)])

BM25 : ['d13', 'd04', 'd12']
Dense: ['d13', 'd04', 'd03']


## 3. Hybrid search: RRF (rank fusion)

RRF uses **rank positions**, not BM25 and cosine scores directly:

$$\mathrm{RRF}(d)=\sum_{r \in \mathrm{ranked\ lists}}\frac{1}{k + \mathrm{rank}_r(d)}$$

Documents absent from a particular list contribute zero. `k=60` is a conventional default, not a universal optimum.

In [ ]:
def rrf_fuse(ranked_lists: list[list[dict]], k: int = 60, top_k: int = 10) -> list[dict]:
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for position, hit in enumerate(ranked, start=1):
            scores[hit["doc_id"]] += 1.0 / (k + position)
    ordered = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)
    return [{"doc_id": doc_id, "score": float(score)} for doc_id, score in ordered[:top_k]]


def hybrid_search(query: str, top_k: int = 10, candidate_k: int = 12) -> list[dict]:
    return rrf_fuse(
        [bm25_search(query, candidate_k), dense_search(query, candidate_k)],
        top_k=top_k,
    )

hybrid_hits = hybrid_search(query, top_k=5)
pd.DataFrame([{**h, "text": DOC_BY_ID[h["doc_id"]]["text"]} for h in hybrid_hits])

,doc_id,score,text
0,d13,0.032787,Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve ...
1,d04,0.032258,Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables sema...
2,d06,0.029877,"A cross-encoder reranker jointly reads a query and a candidate passage, predicts a relevance score, and reorders a s..."
3,d18,0.029631,"A query router selects a data source such as documents, SQL, or web search based on the question. A route must corre..."
4,d08,0.029514,"Parent document retrieval searches short child chunks and returns the larger corresponding parent section, preservin..."


## 4. Cross-encoder reranking

A **bi-encoder** independently embeds queries and documents; a **cross-encoder** reads each `(query, candidate)` pair together. This is slower, so rerank a small candidate set instead of every document.

In [ ]:
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL, device="cpu")


def hybrid_rerank(query: str, top_k: int = 5, candidate_k: int = 12) -> list[dict]:
    candidates = hybrid_search(query, top_k=candidate_k, candidate_k=candidate_k)
    if not candidates:
        return []
    pairs = [(query, DOC_BY_ID[h["doc_id"]]["text"]) for h in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    rescored = [{"doc_id": h["doc_id"], "score": float(s)}
                for h, s in zip(candidates, scores)]
    return sorted(rescored, key=lambda h: h["score"], reverse=True)[:top_k]

# Compare methods on one question (scores have different scales; compare ranks).
methods = {"BM25": bm25_search, "Dense": dense_search,
           "Hybrid (RRF)": hybrid_search, "Hybrid + Rerank": hybrid_rerank}
rows = []
for method_name, fn in methods.items():
    for position, hit in enumerate(fn(query, top_k=3), start=1):
        rows.append({"method": method_name, "rank": position,
                     "doc_id": hit["doc_id"], "text": DOC_BY_ID[hit["doc_id"]]["text"]})
pd.DataFrame(rows)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

,method,rank,doc_id,text
0,BM25,1,d13,Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve ...
1,BM25,2,d04,Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables sema...
2,BM25,3,d12,Original Self-RAG trains a generator to produce retrieval and critique reflection tokens that guide when to retrieve...
3,Dense,1,d13,Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve ...
4,Dense,2,d04,Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables sema...
5,Dense,3,d03,"BM25 is a lexical sparse retriever that scores documents using exact query term matches, term frequency, and documen..."
6,Hybrid (RRF),1,d13,Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve ...
7,Hybrid (RRF),2,d04,Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables sema...
8,Hybrid (RRF),3,d06,"A cross-encoder reranker jointly reads a query and a candidate passage, predicts a relevance score, and reorders a s..."
9,Hybrid + Rerank,1,d13,Multi-query retrieval generates alternative phrasings of the same user question and merges their results to improve ...


## 5. Retrieval evaluation (small, hand-labeled toy set)

**Recall@K:** fraction of known relevant documents found in top K. **MRR@K:** reciprocal rank of the first relevant document. **nDCG@K:** ranking-aware gain, normalized by ideal ordering. These require relevance labels (**qrels**) even though the retrieval models are not trained on this toy dataset.

**Important:** These labels are manually specified for teaching. Do not report these results as BEIR benchmark performance.

In [ ]:
# One or more relevant chunk IDs per query. Labels are binary relevance (0/1).
QRELS = [
    ("What method fuses keyword and semantic rankings without raw score calibration?", {"d05"}),
    ("How does ColBERT match query tokens to document tokens?", {"d09"}),
    ("How can a cross encoder improve the ordering of retrieved passages?", {"d06"}),
    ("Why can HyDE work without task-specific labeled query document pairs?", {"d10"}),
    ("What should RAG do when its retrieved evidence is weak?", {"d11"}),
    ("What is parent document retrieval and why return the parent?", {"d08"}),
    ("How is keyword matching different from semantic vector retrieval?", {"d03", "d04"}),
    ("Which metrics evaluate top K retrieval ranking quality?", {"d07"}),
    ("How are reflection tokens trained in Self-RAG?", {"d12"}),
    ("Why do document chunks sometimes overlap?", {"d02"}),
]


def retrieval_metrics(results: list[dict], gold: set[str], k: int = 3) -> dict:
    ranked = [h["doc_id"] for h in results[:k]]
    if not gold:
        raise ValueError("Each query needs at least one relevant document")
    recall = len(set(ranked) & gold) / len(gold)
    rr = next((1.0 / i for i, doc_id in enumerate(ranked, start=1) if doc_id in gold), 0.0)
    dcg = sum((1.0 / math.log2(i + 2)) for i, doc_id in enumerate(ranked) if doc_id in gold)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(gold))))
    return {"Recall@3": recall, "MRR@3": rr, "nDCG@3": dcg / idcg}


def evaluate_retrievers(methods: dict, qrels=QRELS, k: int = 3) -> pd.DataFrame:
    summary = []
    for name, search_fn in methods.items():
        observed = []
        start = time.perf_counter()
        for question, relevant_ids in qrels:
            observed.append(retrieval_metrics(search_fn(question, top_k=k), relevant_ids, k))
        latency_ms = 1000 * (time.perf_counter() - start) / len(qrels)
        summary.append({"Method": name,
                        **{metric: float(np.mean([row[metric] for row in observed]))
                           for metric in ["Recall@3", "MRR@3", "nDCG@3"]},
                        "Avg latency (ms/query)": latency_ms})
    return pd.DataFrame(summary)

scores = evaluate_retrievers(methods)
display(scores.round(3))

,Method,Recall@3,MRR@3,nDCG@3,Avg latency (ms/query)
0,BM25,0.85,0.8,0.788,0.144
1,Dense,0.95,1.0,0.961,18.993
2,Hybrid (RRF),1.00,0.9,0.918,17.925
3,Hybrid + Rerank,0.95,1.0,0.961,217.848


In [ ]:
# Lightweight correctness checks for the notebook's own algorithms.
assert chunk_words("one two three four five", chunk_size=3, overlap=1) == [
    "one two three", "three four five", "five"
]
assert rrf_fuse([[{"doc_id": "A"}, {"doc_id": "B"}],
                 [{"doc_id": "B"}, {"doc_id": "A"}]])[:2] == [
    {"doc_id": "A", "score": 1 / 61 + 1 / 62},
    {"doc_id": "B", "score": 1 / 62 + 1 / 61},
]
assert retrieval_metrics([{"doc_id": "d05"}], {"d05"})["Recall@3"] == 1.0
print("Sanity checks passed: chunking, RRF, and retrieval metrics.")

Sanity checks passed: chunking, RRF, and retrieval metrics.


## 6. Parent-document retrieval (search small, return larger context)

Here each `parent_id` groups a few related chunks. First retrieve the most relevant **child**, then look up its **parent**. In a real project the parent store might contain the original section or article, not merely concatenated demo chunks.

In [ ]:
PARENTS = defaultdict(list)
for d in DOCS:
    PARENTS[d["parent_id"]].append(d["text"])
PARENTS = {key: "\n".join(texts) for key, texts in PARENTS.items()}


def parent_retrieve(query: str, top_k: int = 2) -> list[dict]:
    results, seen = [], set()
    for hit in hybrid_search(query, top_k=10):
        parent_id = DOC_BY_ID[hit["doc_id"]]["parent_id"]
        if parent_id not in seen:
            seen.add(parent_id)
            results.append({"matched_child": hit["doc_id"], "parent_id": parent_id,
                            "parent_context": PARENTS[parent_id]})
        if len(results) == top_k:
            break
    return results

parent_example = parent_retrieve("How are BM25 and dense retrieval combined?", top_k=1)[0]
print("Matched child:", parent_example["matched_child"])
print("Returned parent context:\n", parent_example["parent_context"])

Matched child: d04
Returned parent context:
 BM25 is a lexical sparse retriever that scores documents using exact query term matches, term frequency, and document length normalization.
Dense retrieval uses a pretrained embedding model to encode queries and passages as vectors. Similarity enables semantic matching beyond exact words.


## 7. Grounded answer generation (optional local LLM)

The retrieval pipeline already works without an API. Below is a real **RAG prompt** made from retrieved passages. You can optionally generate an answer with a **local open model** (`google/flan-t5-small`) by changing `RUN_LOCAL_GENERATION = True`. It downloads another model, uses CPU, and **may produce weak answers**; this is a mechanics demo, not a trustworthy QA system.

Citations shown by a model must still be checked against the actual source text.

In [ ]:
def build_grounded_prompt(question: str, top_k: int = 3) -> tuple[str, list[dict]]:
    hits = hybrid_rerank(question, top_k=top_k)
    evidence = "\n".join(
        f'[{hit["doc_id"]}] {DOC_BY_ID[hit["doc_id"]]["text"]}'
        for hit in hits
    )
    prompt = (
        "Answer the question using ONLY the evidence below. "
        "If the evidence is insufficient, say 'I do not know based on these sources.' "
        "Be concise and cite supporting document IDs in brackets.\n\n"
        f"Evidence:\n{evidence}\n\nQuestion: {question}\nAnswer:"
    )
    return prompt, hits

question = "What is reciprocal rank fusion?"
prompt, supporting_hits = build_grounded_prompt(question)
print(prompt)

Answer the question using ONLY the evidence below. If the evidence is insufficient, say 'I do not know based on these sources.' Be concise and cite supporting document IDs in brackets.

Evidence:
[d05] Reciprocal Rank Fusion (RRF) combines multiple ranked lists using the reciprocal of each document's rank. It does not need comparable raw scores.
[d06] A cross-encoder reranker jointly reads a query and a candidate passage, predicts a relevance score, and reorders a small retrieved candidate set.
[d07] Recall at K measures how many relevant documents are retrieved; MRR measures the first relevant rank; nDCG at K rewards relevant documents near the top.

Question: What is reciprocal rank fusion?
Answer:


In [ ]:
RUN_LOCAL_GENERATION = True  # Switch to True for an entirely API-free local LLM call.

if RUN_LOCAL_GENERATION:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    generator_id = "google/flan-t5-small"
    gen_tokenizer = AutoTokenizer.from_pretrained(generator_id)
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(generator_id)
    gen_model.eval()
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.inference_mode():
        generated = gen_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    print(gen_tokenizer.decode(generated[0], skip_special_tokens=True))
else:
    print("Local generation skipped. Set RUN_LOCAL_GENERATION=True to run FLAN-T5 (no API key).")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

combines multiple ranked lists using the reciprocal of each document's rank


## 8. One tiny ColBERT concept check: MaxSim

ColBERT is **token-level late interaction**, not single-vector cosine search. For every query-token vector, find the document-token vector with the highest similarity, then sum. These are **toy numeric vectors**, not actual trained ColBERT embeddings.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

q_tokens = np.array([[0.8, 0.2, 0.1], [0.1, 0.9, 0.2]])
d_tokens = np.array([[0.2, 0.9, 0.1], [0.6, 0.3, 0.2], [0.8, 0.1, 0.1]])
sim_matrix = cosine_similarity(q_tokens, d_tokens)
maxsim = sim_matrix.max(axis=1).sum()
print("Token similarity matrix shape:", sim_matrix.shape)
print("MaxSim score:", round(float(maxsim), 3))

Token similarity matrix shape: (2, 3)
MaxSim score: 1.981


## 9. Advanced RAG cheat sheet (no extra code required)

| Technique | One-line intuition | Requires paid API? | In this notebook |
|---|---|---|---|
| **Multi-query / Decomposition** | Rewrite one query or split a complex question; avoid needless expansion | No; typically needs an LLM, local is possible | Concept |
| **HyDE** | Generate a hypothetical answer, embed it, and retrieve real documents | No if local LLM; otherwise provider dependent | Concept |
| **Parent / Multi-representation** | Search a smaller/simpler representation, return original context | No | Parent demo |
| **RAPTOR** | Cluster and summarize passages recursively for multi-level retrieval | No if local summarization | Concept |
| **ColBERT / MaxSim** | Match individual contextualized token embeddings | No if locally hosted | Toy MaxSim |
| **CRAG** | Grade retrieved evidence and correct weak retrieval before generation | No if local models; web search may require service | Concept |
| **Original Self-RAG** | Fine-tuned model generates special retrieval / critique tokens | No API required if self-hosted, but requires specialized training/model | Concept |
| **Vector database** | Efficient nearest-neighbor index for a large corpus | No: FAISS is local; hosted services may charge | Exact cosine on tiny corpus |

**Original Self-RAG ≠ a LangGraph workflow with yes/no graders.** The latter is a *Self-RAG-like control flow*, not a demonstration of learned reflection tokens.

**Do not copy the source document's unexplained numeric gains** (latencies, BEIR scores, hallucination rates) into a README as verified results. Evaluate your own corpus and document the benchmark setup.

**Production checklist:** source permissions/privacy; robust parsing and chunk IDs; metadata filters; proper ANN index; citations linked to source spans; failure/abstention handling; held-out qrels; cost, latency, and safety monitoring.

### References / libraries

- [Sentence Transformers: Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- [Sentence Transformers: Cross-Encoder](https://www.sbert.net/docs/cross_encoder/usage/usage.html)
- [rank-bm25](https://github.com/dorianbrown/rank_bm25)
- [Hugging Face FLAN-T5 small](https://huggingface.co/google/flan-t5-small)
- [Hugging Face RAG Evaluation Cookbook](https://huggingface.co/learn/cookbook/en/rag_evaluation)

### How to turn this into a portfolio project

1. Replace the toy corpus with public GitHub Issues/docs or your own permitted data.
2. Create 50–100 **held-out queries** and judge relevant document IDs; show annotation guidelines.
3. Report Recall@K, nDCG@K, MRR, p50/p95 latency, hardware and exact model versions.
4. Add `pytest` for RRF and metrics, save environment versions, and link the notebook in your GitHub README.